# Benchmark error analysis

Companion to the Streamlit viewer. Streamlit shows aggregates; this notebook is for **reading individual posts** where the model gets it wrong.

Run order:
1. `benchmark embed && benchmark run configs/<file>.yaml && benchmark report` (in `apps/benchmark/`)
2. Open this notebook and Restart & Run All

Each section after Setup is independent — pick the ones you need.

## 0. Setup

Loads `summary.parquet` plus every per-experiment parquet into a `runs` dict. Sets `DATA_PATH` and `chosen_experiment` — change `chosen_experiment` to drill into a different run.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK = Path.cwd()
APP_DIR = NOTEBOOK.parent if NOTEBOOK.name == "notebooks" else NOTEBOOK
LLM_DIR = APP_DIR.parent / "llm"
for p in (str(APP_DIR), str(LLM_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import plotly.express as px

RESULTS_DIR = APP_DIR / "results"
DATA_PATH = APP_DIR / "data" / "control.jsonl"

summary_path = RESULTS_DIR / "summary.parquet"
if not summary_path.exists():
    print(f"no summary at {summary_path} - run `benchmark report` first")
    summary = pd.DataFrame()
    runs: dict[str, pd.DataFrame] = {}
else:
    summary = pd.read_parquet(summary_path)
    runs = {
        p.stem: pd.read_parquet(p)
        for p in sorted(RESULTS_DIR.glob("*.parquet"))
        if p.name != "summary.parquet"
    }
    print(f"loaded {len(summary)} experiments, {len(runs)} per-experiment parquets")

# Default drill-down target. Change this to inspect a different experiment.
chosen_experiment = next(iter(runs)) if runs else None
print(f"chosen_experiment = {chosen_experiment!r}")

summary

In [ ]:
# Helper: attach the original post text to a results frame so we can read what we got wrong.
def attach_text(df: pd.DataFrame) -> pd.DataFrame:
    if not DATA_PATH.exists():
        return df
    ds = pd.read_json(DATA_PATH, lines=True)[["post_id", "text"]]
    return df.merge(ds, on="post_id", how="left")

def relevant_rows(df: pd.DataFrame) -> pd.DataFrame:
    return df[(df["pred_relevant"] == True) & df["error"].isna()]

# Sanity: how many predictions and errors per experiment?
if runs:
    health = pd.DataFrame([
        {
            "experiment_id": eid,
            "n": len(df),
            "n_predicted": int(len(relevant_rows(df))),
            "n_errors": int(df["error"].notna().sum()),
            "n_irrelevant": int((df["pred_relevant"] == False).sum()),
        }
        for eid, df in runs.items()
    ])
    display(health)

## 1. Score-regression error analysis

Signed-error histogram (positive = model overshoots), then the worst N misses with the actual post text.

In [ ]:
rows = []
for eid, df in runs.items():
    rel = relevant_rows(df).dropna(subset=["pred_score"])
    rows.extend(
        {"experiment_id": eid, "signed_error": float(p - g)}
        for p, g in zip(rel["pred_score"], rel["gt_score"])
    )
err_df = pd.DataFrame(rows)
if err_df.empty:
    print("no scored predictions yet")
else:
    fig = px.histogram(
        err_df, x="signed_error", color="experiment_id",
        barmode="overlay", nbins=30,
        title="Signed score errors (pred - gt). Right of zero = overshoot.",
    )
    fig.show()

In [ ]:
from benchmark.metrics.regression import regression_report

N_WORST = 10
if chosen_experiment:
    df = runs[chosen_experiment]
    rel = attach_text(relevant_rows(df).dropna(subset=["pred_score"]).copy())
    rel["abs_err"] = (rel["pred_score"] - rel["gt_score"]).abs()

    rep = regression_report(rel["gt_score"], rel["pred_score"])
    print(f"{chosen_experiment}: MAE={rep.mae:.3f} RMSE={rep.rmse:.3f} "
          f"mean_error={rep.mean_error:+.3f} pearson_r={rep.pearson_r:.3f}")
    cols = [c for c in ["post_id", "text", "gt_score", "pred_score", "abs_err"] if c in rel.columns]
    display(rel.nlargest(N_WORST, "abs_err")[cols])

## 2. Emotion confusion drill-down

Read the actual posts where the model got the emotion wrong. Look for systematic patterns (e.g. SADNESS classified as ANGER on customer-service complaints).

In [ ]:
if chosen_experiment:
    df = runs[chosen_experiment]
    rel = attach_text(relevant_rows(df).dropna(subset=["pred_emotion"]).copy())
    wrong = rel[rel["pred_emotion"] != rel["gt_emotion"]]
    print(f"{len(wrong)}/{len(rel)} emotion mistakes in {chosen_experiment}")
    cols = [c for c in ["post_id", "text", "gt_emotion", "pred_emotion", "gt_score", "pred_score"] if c in wrong.columns]
    display(wrong[cols].head(30))

## 3. Aspect mismatch (raw vs normalized)

When the model's raw aspect output isn't in our taxonomy, `metrics/aspect.py::_SYNONYMS` maps it (e.g. "shipping" → DELIVERY). Mistakes here come in two flavors:

- The model is genuinely wrong (raw and normalized both don't match GT).
- Synonym map is incomplete — extend `_SYNONYMS` if you see common phrasings landing in OTHER.

In [ ]:
if chosen_experiment:
    df = runs[chosen_experiment]
    rel = attach_text(relevant_rows(df).dropna(subset=["pred_aspect_normalized"]).copy())
    wrong = rel[rel["pred_aspect_normalized"] != rel["gt_aspect"]]
    print(f"{len(wrong)}/{len(rel)} aspect mistakes (after normalization)")
    cols = [c for c in ["post_id", "text", "gt_aspect", "pred_aspect_raw", "pred_aspect_normalized"] if c in wrong.columns]
    display(wrong[cols].head(30))

    # Which raw outputs land in OTHER? Candidates for the synonym map.
    from benchmark.metrics.aspect import OTHER
    in_other = rel[rel["pred_aspect_normalized"] == OTHER]
    if not in_other.empty:
        print(f"\nraw outputs landing in OTHER ({len(in_other)} rows) - candidates for _SYNONYMS:")
        display(in_other["pred_aspect_raw"].value_counts().head(15))

## 4. Retrieved-exemplar inspection

For few-shot-retrieved experiments, read the exemplars the embedder picked. The failure mode you're hunting for: high lexical overlap but wrong sentiment (e.g. retrieves "loved this product" for a sarcastic "oh I just LOVED waiting on hold").

In [ ]:
def has_fewshot(df: pd.DataFrame) -> bool:
    if "fewshot_post_ids" not in df.columns:
        return False
    return df["fewshot_post_ids"].apply(lambda x: len(x) > 0 if x is not None else False).any()

fewshot_runs = {eid: df for eid, df in runs.items() if has_fewshot(df)}
if not fewshot_runs:
    print("no few-shot retrieved runs found - this section is a no-op")
elif not DATA_PATH.exists():
    print(f"need {DATA_PATH} to look up exemplar text")
else:
    fs_eid = next(iter(fewshot_runs))
    df = fewshot_runs[fs_eid]
    rel = relevant_rows(df)
    if rel.empty:
        print("no relevant rows in this experiment")
    else:
        # Pick the worst score-miss in this experiment - that's where retrieval
        # quality matters most.
        rel = rel.dropna(subset=["pred_score"]).copy()
        rel["abs_err"] = (rel["pred_score"] - rel["gt_score"]).abs()
        target = rel.nlargest(1, "abs_err").iloc[0]
        ds = pd.read_json(DATA_PATH, lines=True).set_index("post_id")

        print(f"=== experiment {fs_eid}, post {int(target['post_id'])} ===")
        print(f"text: {ds.loc[target['post_id'], 'text']}")
        print(f"gt:   score={ds.loc[target['post_id'], 'gt_score']} "
              f"emotion={ds.loc[target['post_id'], 'gt_emotion']} "
              f"aspect={ds.loc[target['post_id'], 'gt_aspect']}")
        print(f"pred: score={target['pred_score']} "
              f"emotion={target['pred_emotion']} "
              f"aspect={target['pred_aspect_normalized']}")
        print("\nretrieved exemplars (in rank order):")
        for ex_id in target["fewshot_post_ids"]:
            ex = ds.loc[int(ex_id)]
            print(f"  [{ex_id}] {ex['text']!r}")
            print(f"        gt: score={ex['gt_score']} emotion={ex['gt_emotion']} aspect={ex['gt_aspect']}")

## 5. Per-fold breakdown

If one fold is markedly worse than the others, suspect either dataset imbalance or a bug in the leakage masking. Both worth knowing before paper claims.

In [ ]:
rows = []
for eid, df in runs.items():
    rel = relevant_rows(df).dropna(subset=["pred_score"])
    for fold, sub in rel.groupby("fold"):
        if sub.empty:
            continue
        rep = regression_report(sub["gt_score"], sub["pred_score"])
        emo_acc = (
            float((sub["pred_emotion"] == sub["gt_emotion"]).mean())
            if "pred_emotion" in sub.columns else float("nan")
        )
        rows.append({
            "experiment_id": eid,
            "fold": int(fold),
            "n": len(sub),
            "score_mae": rep.mae,
            "emotion_accuracy": emo_acc,
        })
fold_df = pd.DataFrame(rows)
if fold_df.empty:
    print("no per-fold data")
else:
    display(fold_df.sort_values(["experiment_id", "fold"]).reset_index(drop=True))
    fig = px.bar(
        fold_df, x="fold", y="score_mae", color="experiment_id",
        barmode="group", title="Per-fold score MAE",
    )
    fig.show()

## 6. Pairwise disagreement browser

Pick two experiments, read the rows where they disagree. The Streamlit viewer gives you significance numbers; this gives you the actual posts so you can write qualitative observations for the paper.

In [ ]:
exp_ids = list(runs)
if len(exp_ids) < 2:
    print(f"need >= 2 experiments, have {len(exp_ids)}")
else:
    a, b = exp_ids[0], exp_ids[1]  # change to compare different pairs
    da, db = runs[a], runs[b]
    merged = da.merge(db, on="post_id", suffixes=("_a", "_b"))
    merged = merged[
        (merged["pred_relevant_a"] == True) & (merged["pred_relevant_b"] == True)
        & merged["error_a"].isna() & merged["error_b"].isna()
    ]
    merged = attach_text(merged) if "text" not in merged.columns else merged
    disagree = merged[
        (merged["pred_emotion_a"] != merged["pred_emotion_b"])
        | (merged["pred_aspect_normalized_a"] != merged["pred_aspect_normalized_b"])
    ]
    print(f"{a} vs {b}: {len(disagree)}/{len(merged)} posts disagree")
    cols = [c for c in [
        "post_id", "text",
        "gt_emotion_a", "pred_emotion_a", "pred_emotion_b",
        "gt_aspect_a", "pred_aspect_normalized_a", "pred_aspect_normalized_b",
        "gt_score_a", "pred_score_a", "pred_score_b",
    ] if c in disagree.columns]
    display(disagree[cols].head(20))

## 7. Latency distribution

Cost story isn't just MAE. Once paid RunPod runs start, p95 latency is part of the comparison.

In [ ]:
frames = []
for eid, df in runs.items():
    sub = df[df["latency_ms"].notna()][["latency_ms"]].copy()
    sub["experiment_id"] = eid
    frames.append(sub)
if not frames:
    print("no latency data (cached results don't re-time)")
else:
    lat_df = pd.concat(frames, ignore_index=True)
    fig = px.box(
        lat_df, x="experiment_id", y="latency_ms",
        points="outliers", title="Latency by experiment (log scale)",
    )
    fig.update_yaxes(type="log")
    fig.show()
    display(lat_df.groupby("experiment_id")["latency_ms"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))